# GAN Augmentation in Low-Data Regimes: a Controlled Multi-Dataset Study
### ISE1 Course Seminar, Fr. CRCE | Experiment notebook v3

**Question.** Under a fair, controlled protocol, when does class-conditional WGAN-GP augmentation help low-data image classification, compared with no augmentation and with classical augmentation? How does the answer depend on dataset type, amount of real data, classifier, and synthetic ratio?

**Datasets**
- **Brain Tumor MRI** (Kaggle, grayscale, 4 classes): official `Testing` folder is the test set.
- **PlantVillage subset** (Apple, Corn, Grape, Potato, colour, 15 classes): there is no official split, so a fixed, balanced test set is held out (30 images per class, fixed seed).

**Classifiers.** ResNet-18 (full grid) and DenseNet-121 (smaller grid, set in `clf_plan`), both ImageNet-pretrained.

**Arms.** No augmentation, classical augmentation, WGAN-GP at 25/50/100% synthetic, and WGAN-GP 50% + classical. The main GAN comparison (50%) is fixed in advance, so no ratio is picked using test accuracy.

**Two image pipelines.** `native` keeps real images sharp. `matched64` pushes real, synthetic and test images through the same 64px-then-upscale path. If a GAN only looks bad under `native`, part of the loss was a blur shortcut, not the augmentation.

**Everything lives on Google Drive** (`MyDrive/seminar_gan_project/v3/`). Every finished GAN and every finished classifier run is its own small file. If Colab disconnects, just press Run all again: finished work is skipped and half-trained GANs resume from their last checkpoint. A progress table (section 8) shows what is done and what is left at any time.

**How to run, in order**
1. Runtime > Change runtime type > GPU.
2. Set `PILOT = True` in section 2 for a quick check on MRI only (1 seed, regimes 100 and 20, ResNet-18). Run all. Look at the data report and the first table.
3. Set `PILOT = False`. Run all again. Use `RUN_DATASETS` to run one dataset at a time and `MAX_RUNTIME_MIN` to stop cleanly before a Colab session limit.
4. Fortis clinical review is **optional** and off by default (section 12).

**What changed from v1, and why.** GAN trained for a fixed 10,000 critic steps (v1 got only 120 to 720 generator updates). Seeds change the real subset, the GAN and the classifier. Equal optimizer steps across arms. Leakage audit (exact and near duplicates). KID with a real-vs-real floor instead of FID (FID is invalid at 80 to 1500 images). Paired bootstrap confidence intervals and Holm correction instead of Wilcoxon at n=3. Generator BatchNorm uses the default eps (v1 accidentally used 0.8).

**Known limitations to state in any write-up.** PlantVillage contains several photos of the same leaf, so some near duplicates survive that a hash cannot catch. 64px GANs are weak by modern standards. Seed counts are 5 (MRI) and 3 (PlantVillage), so p-values are coarse and the confidence intervals are the main evidence.

## 1. Setup

In [ ]:
import os, sys, io, json, time, math, random, hashlib, shutil, zipfile, platform, subprocess, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from torchvision.transforms import v2
from PIL import Image
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import (precision_recall_fscore_support, confusion_matrix, roc_auc_score,
                             matthews_corrcoef, cohen_kappa_score, balanced_accuracy_score, accuracy_score)
try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore")
SMOKE = os.environ.get("SEMINAR_SMOKE", "0") == "1"      # developer test flag, leave it alone
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE,
      "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "no GPU")
if DEVICE.type != "cuda" and not SMOKE:
    raise RuntimeError("No GPU attached. Runtime > Change runtime type > GPU, then run again.")
torch.backends.cudnn.benchmark = True

## 2. Configuration (single source of truth)

In [ ]:
PILOT = False                 # True: MRI only, 1 seed, regimes 100 and 20, ResNet-18. Results are reused by the full run.
RUN_DATASETS = ["brain_mri", "plantvillage"]   # run one at a time if you want, e.g. ["brain_mri"]
MAX_RUNTIME_MIN = None        # e.g. 170: stop cleanly after this many minutes (Colab sessions end). None = no limit.

CFG = dict(
    datasets=["brain_mri", "plantvillage"],
    ds_seeds={"brain_mri": [0, 1, 2, 3, 4], "plantvillage": [0, 1, 2]},
    clf_plan={                                           # which classifier runs which regimes and seeds
        "resnet18":    {"regimes": [100, 60, 40, 20], "seeds": None},     # None = all seeds of that dataset
        "densenet121": {"regimes": [100, 20],         "seeds": [0, 1, 2]},
    },
    pipelines=["native", "matched64"],
    ratios=[0.25, 0.50, 1.00],            # synthetic added, as a fraction of real images per class
    combo_ratios=[0.50],                  # ratios also run together with classical augmentation
    primary_ratio=0.50,                   # fixed in advance, used for headline comparisons
    gan_variants=["wgangp"],              # add "wgangp_da" later for the DiffAugment variant
    # GAN
    gan_img=64, latent=100, gan_iters=10000, gan_bs=32, gan_lr=1e-4, n_critic=5,
    lambda_gp=10.0, gan_ckpt_every=1000,
    # classifier
    clf_img=224, clf_epochs_base=15, clf_bs=16, clf_lr=1e-4,
    equal_steps=True, amp=True, pretrained=True,
    # data hygiene
    near_dup_bits=4, subset_version="nested_v2", pv_test_per_class=30,
)
if PILOT:
    RUN_DATASETS = ["brain_mri"]
    CFG["ds_seeds"]["brain_mri"] = [0]
    CFG["clf_plan"] = {"resnet18": {"regimes": [100, 20], "seeds": None}}
if SMOKE:
    CFG.update(ds_seeds={"brain_mri": [0, 1], "plantvillage": [0]},
               clf_plan={"resnet18": {"regimes": [6, 3], "seeds": None},
                         "densenet121": {"regimes": [3], "seeds": [0]}},
               gan_iters=20, gan_ckpt_every=10, clf_img=64, clf_epochs_base=2, clf_bs=4,
               pretrained=False, amp=False, pv_test_per_class=3)

IMG, GIMG = CFG["clf_img"], CFG["gan_img"]
ALL_REGIMES = sorted({r for p in CFG["clf_plan"].values() for r in p["regimes"]}, reverse=True)
PRIMARY_VARIANT = CFG["gan_variants"][0]
def arm_name(variant, ratio, combo=False):
    return f"gan-{variant}-r{int(round(ratio * 100))}" + ("+cls" if combo else "")
PRIMARY = arm_name(PRIMARY_VARIANT, CFG["primary_ratio"])
PRIMARY_COMBO = arm_name(PRIMARY_VARIANT, CFG["primary_ratio"], True)

def _h(d): return hashlib.md5(json.dumps(d, sort_keys=True).encode()).hexdigest()[:8]
GAN_KEYS = ["gan_img", "latent", "gan_iters", "gan_bs", "gan_lr", "n_critic", "lambda_gp",
            "subset_version", "near_dup_bits", "pv_test_per_class"]
CLF_KEYS = GAN_KEYS + ["clf_img", "clf_epochs_base", "clf_bs", "clf_lr", "equal_steps", "amp", "pretrained"]
GHASH = _h({k: CFG[k] for k in GAN_KEYS})
PHASH = _h({k: CFG[k] for k in CLF_KEYS})     # regimes, seeds and classifier plan are NOT in the hash, so extending them reuses old results
print("regimes:", ALL_REGIMES, "| primary GAN arm:", PRIMARY, "| GAN hash", GHASH, "| protocol hash", PHASH)

## 3. Google Drive and output folders

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/seminar_gan_project")
else:
    DRIVE_ROOT = Path(os.environ.get("SEMINAR_ROOT", "./seminar_gan_project")).resolve()

OUT = DRIVE_ROOT / "v3"
FIG_DIR, TAB_DIR, LOG_DIR = OUT / "figures", OUT / "tables", OUT / "logs"
LOCAL = Path("/content/local_data") if IN_COLAB else OUT / "_local_cache"     # fast local disk copy for reading images
def ds_dirs(ds):
    base = OUT / "datasets" / ds
    return base / "gan" / GHASH, base / "runs" / PHASH, base / "kid" / GHASH
for d in (OUT, FIG_DIR, TAB_DIR, LOG_DIR, LOCAL):
    d.mkdir(parents=True, exist_ok=True)
for ds in CFG["datasets"]:
    for d in ds_dirs(ds): d.mkdir(parents=True, exist_ok=True)

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def _np(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    raise TypeError(type(o))

def save_json(obj, path):
    tmp = str(path) + ".tmp"
    with open(tmp, "w") as f: json.dump(obj, f, indent=1, default=_np)
    os.replace(tmp, path)

def atomic_torch_save(obj, path):
    tmp = str(path) + ".tmp"; torch.save(obj, tmp); os.replace(tmp, path)

def log(msg):
    print(msg)
    with open(LOG_DIR / "run_log.txt", "a") as f: f.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')} {msg}\n")

# Drive write/read health check, so a broken mount fails now and not 3 hours in
_t = OUT / ".drive_check"; _t.write_text("ok"); assert _t.read_text() == "ok"; _t.unlink()
try:
    du = shutil.disk_usage(str(DRIVE_ROOT)); print(f"Drive check passed. Free space: {du.free / 1e9:.1f} GB")
except Exception:
    print("Drive check passed.")
save_json(dict(cfg=CFG, ghash=GHASH, phash=PHASH, torch=torch.__version__, python=platform.python_version(),
               gpu=torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu"), OUT / "env_and_config.json")
print("All outputs go to:", OUT)

## 4. Datasets: find on Drive, otherwise download once and keep a copy on Drive

In [ ]:
MRI_CLASSES = ["glioma", "meningioma", "notumor", "pituitary"]
PV_PREFIXES = ("apple", "corn", "grape", "potato")            # Apple, Corn (maize), Grape, Potato: 4+4+4+3 = 15 classes
MRI_OFFICIAL_TEST = 1311
DATA_MRI = DRIVE_ROOT / "brain_tumor_mri"
DATA_PV = DRIVE_ROOT / "plantvillage"

def pv_class_dirs(base):
    base = Path(base)
    if not base.is_dir(): return []
    return sorted([d for d in base.iterdir() if d.is_dir() and d.name.lower().startswith(PV_PREFIXES)])
def ok_mri(r):
    r = Path(r); return all((r / sp / c).is_dir() for sp in ("Training", "Testing") for c in MRI_CLASSES)
def ok_pv(r): return len(pv_class_dirs(r)) == 15

def find_root(bases, check, depth=3):
    for base in bases:
        base = Path(base)
        if not base.exists(): continue
        if check(base): return base
        level = [base]
        for _ in range(depth):
            nxt = []
            for b in level:
                try: nxt += [p for p in b.iterdir() if p.is_dir()]
                except Exception: pass
            for p in nxt:
                if check(p): return p
            level = nxt
    return None

def kagglehub_download(slug):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
    import kagglehub
    return Path(kagglehub.dataset_download(slug))      # public dataset, no API key needed

def ensure_mri():
    found = find_root([DATA_MRI, DRIVE_ROOT, Path("/content")], ok_mri)
    if found is None:
        p = kagglehub_download("masoudnickparvar/brain-tumor-mri-dataset")
        src = find_root([p], ok_mri, depth=4)
        assert src is not None, "MRI download has an unexpected layout: " + str(p)
        shutil.copytree(src, DATA_MRI, dirs_exist_ok=True); found = DATA_MRI      # keep a copy on Drive
    return found

def ensure_pv():
    found = find_root([DATA_PV, DRIVE_ROOT, Path("/content")], ok_pv)
    if found is None:
        p = kagglehub_download("abdallahalidev/plantvillage-dataset")
        cands = [d for d in p.rglob("*") if d.is_dir() and ok_pv(d)]
        assert cands, "PlantVillage download has an unexpected layout: " + str(p)
        src = next((d for d in cands if d.name.lower() == "color"), cands[0])      # prefer the colour version
        DATA_PV.mkdir(parents=True, exist_ok=True)
        for d in pv_class_dirs(src): shutil.copytree(d, DATA_PV / d.name, dirs_exist_ok=True)   # only the 15 classes we use
        found = DATA_PV
    return found

def to_local(src, name):
    """Copy to fast local disk once per session; reading thousands of files straight from Drive is slow."""
    dst = LOCAL / name
    if dst.exists() and any(dst.iterdir()): return dst
    if SMOKE and not IN_COLAB: return Path(src)
    shutil.copytree(src, dst, dirs_exist_ok=True); return dst
print("helpers ready")

In [ ]:
EXTS = {".jpg", ".jpeg", ".png"}
def list_dirs(class_dirs):
    files, labels = [], []
    for ci, d in enumerate(class_dirs):
        for p in sorted(Path(d).iterdir()):
            if p.suffix.lower() in EXTS: files.append(str(p)); labels.append(ci)
    return files, np.array(labels)

def load_one(args):
    path, ch = args
    with open(path, "rb") as fh: raw = fh.read()
    md5 = hashlib.md5(raw).hexdigest()
    img = Image.open(io.BytesIO(raw)); g = img.convert("L")
    base = g if ch == 1 else img.convert("RGB")
    def arr(im, s):
        a = np.asarray(im.resize((s, s), Image.BILINEAR), dtype=np.uint8)
        return a[None] if ch == 1 else a.transpose(2, 0, 1)
    d = np.asarray(g.resize((17, 16), Image.BILINEAR), dtype=np.int16)                 # 256-bit difference hash
    bits = np.packbits((d[:, 1:] > d[:, :-1]).flatten())
    return md5, arr(base, IMG), arr(base, GIMG), bits

def load_raw(class_dirs, ch, name):
    files, y = list_dirs(class_dirs); t = time.time()
    with ThreadPoolExecutor(max_workers=8) as ex: res = list(ex.map(load_one, [(f, ch) for f in files]))
    raw = dict(y=y, md5=np.array([r[0] for r in res]), x_img=np.stack([r[1] for r in res]),
               x64=np.stack([r[2] for r in res]), bits=np.stack([r[3] for r in res]))
    del res
    print(f"{name}: {len(files)} images in {time.time() - t:.0f}s | per class {np.bincount(y).tolist()}")
    return raw

def take(raw, idx): return {k: v[idx] for k, v in raw.items()}

POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
def min_hamming(a_bits, b_bits, chunk=256):
    out = np.empty(len(a_bits), dtype=np.int32)
    for i in range(0, len(a_bits), chunk):
        x = np.bitwise_xor(a_bits[i:i + chunk, None, :], b_bits[None, :, :])
        out[i:i + chunk] = POP[x].sum(-1).min(1)
    return out

def audit(raw_tr, raw_te, nc):
    """Remove train images that duplicate each other or are exact/near duplicates of a test image."""
    tr_md5, te_md5 = raw_tr["md5"], raw_te["md5"]
    exact = np.isin(tr_md5, te_md5)
    _, first = np.unique(tr_md5, return_index=True)
    is_first = np.zeros(len(tr_md5), bool); is_first[first] = True
    near = min_hamming(raw_tr["bits"], raw_te["bits"]) <= CFG["near_dup_bits"]
    keep = is_first & ~near
    rep = dict(train_per_class=np.bincount(raw_tr["y"], minlength=nc).tolist(),
               test_per_class=np.bincount(raw_te["y"], minlength=nc).tolist(),
               exact_duplicates_train_vs_test=int(exact.sum()), near_duplicates_train_vs_test=int(near.sum()),
               near_threshold_bits_of_256=CFG["near_dup_bits"],
               exact_duplicates_within_train=int((~is_first).sum()),
               exact_duplicates_within_test=int(len(te_md5) - len(np.unique(te_md5))), pool_size=int(keep.sum()))
    pool = take(raw_tr, np.where(keep)[0])
    rep["pool_per_class"] = np.bincount(pool["y"], minlength=nc).tolist()
    return pool, raw_te, rep

DATASETS = {}      # cache of loaded datasets (RAM)
def get_dataset(name):
    if name in DATASETS: return DATASETS[name]
    if name == "brain_mri":
        root = to_local(ensure_mri(), "brain_tumor_mri"); classes = MRI_CLASSES; ch = 1
        raw_tr = load_raw([root / "Training" / c for c in classes], ch, "MRI Training")
        raw_te = load_raw([root / "Testing" / c for c in classes], ch, "MRI Testing")
        note = None
        if sum(np.bincount(raw_te["y"])) != MRI_OFFICIAL_TEST:
            note = (f"Test set has {len(raw_te['y'])} images, the official Kaggle Testing split has {MRI_OFFICIAL_TEST}. "
                    "Report the exact count and where your copy came from.")
    elif name == "plantvillage":
        root = to_local(ensure_pv(), "plantvillage"); cdirs = pv_class_dirs(root); classes = [d.name for d in cdirs]; ch = 3
        print("PlantVillage classes:", classes)
        raw = load_raw(cdirs, ch, "PlantVillage")
        rng = np.random.RandomState(777); _, first = np.unique(raw["md5"], return_index=True)
        isf = np.zeros(len(raw["y"]), bool); isf[first] = True; te = np.zeros(len(raw["y"]), bool)
        for c in range(len(classes)):
            cand = np.where((raw["y"] == c) & isf)[0]; rng.shuffle(cand); te[cand[:CFG["pv_test_per_class"]]] = True
        raw_tr, raw_te = take(raw, np.where(~te)[0]), take(raw, np.where(te)[0]); del raw
        note = "PlantVillage has several photos of the same leaf; near duplicates a hash cannot catch may remain across the split."
    else:
        raise ValueError(name)
    nc = len(classes)
    pool, test, rep = audit(raw_tr, raw_te, nc); del raw_tr
    rep.update(dataset=name, classes=classes, note=note, max_regime=int(min(rep["pool_per_class"])))
    save_json(rep, OUT / f"data_report_{name}.json")
    print(f"\n=== data report: {name} ===")
    for k, v in rep.items(): print(f"  {k:34s} {v}")
    DATASETS[name] = dict(name=name, classes=classes, nc=nc, ch=ch, pool=pool, test=test, report=rep)
    return DATASETS[name]
print("dataset loaders ready (nothing is loaded until it is needed)")

### 4.1 Per-dataset state: subsets, image pipelines, test tensors

In [ ]:
CUR = {"name": None}
def make_subsets(seed):
    """Per seed, a random class-balanced subset. Regimes are nested (20 inside 40 inside 60 inside 100)."""
    rng = np.random.RandomState(10_000 + seed); y = POOL["y"]
    perms = [rng.permutation(np.where(y == c)[0]) for c in range(NC)]
    return {n: np.concatenate([p[:n] for p in perms]) for n in ALL_REGIMES if n <= min(len(p) for p in perms)}

def to_u8_img(x01):
    """float [0,1] (n,C,h,w) -> uint8 (n,C,IMG,IMG), bilinear. Shared by synthetic and matched real images."""
    x = F.interpolate(x01, size=(IMG, IMG), mode="bilinear", align_corners=False)
    return (x.clamp(0, 1) * 255).round().to(torch.uint8)

def u8_64_to_img(a64):
    return to_u8_img(torch.as_tensor(np.asarray(a64), device=DEVICE).float().div(255))

def synth_to_img(g):
    """GAN output in [-1,1] -> 8-bit -> same upsampling path as matched real images."""
    return to_u8_img(((g * 0.5 + 0.5).clamp(0, 1) * 255).round() / 255)

def real_images(idx, pipe):
    if pipe == "native": return torch.as_tensor(POOL["x_img"][idx], device=DEVICE)
    return u8_64_to_img(POOL["x64"][idx])

def build_test(pipe):
    if pipe == "native": return torch.as_tensor(TEST["x_img"], device=DEVICE)
    return torch.cat([u8_64_to_img(TEST["x64"][i:i + 256]) for i in range(0, len(TEST["x64"]), 256)])

def use_dataset(D):
    """Bind the globals every other function reads. Call before anything that touches data."""
    g = globals()
    g.update(CLASSES=D["classes"], NC=D["nc"], CH=D["ch"], POOL=D["pool"], TEST=D["test"], DS=D["name"])
    g["GAN_DIR"], g["RUNS_DIR"], g["KID_DIR"] = ds_dirs(D["name"])
    g["TEST_X"] = {p: build_test(p) for p in CFG["pipelines"]}
    g["TEST_Y"] = torch.as_tensor(TEST["y"], device=DEVICE)
    CUR["name"] = D["name"]
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    print(f"[dataset: {D['name']}] classes={NC} channels={CH} pool={len(POOL['y'])} test={len(TEST['y'])} max regime={D['report']['max_regime']}")

## 5. Class-conditional WGAN-GP (optional DiffAugment variant)

In [ ]:
class Generator(nn.Module):
    def __init__(self, nc, ch, latent=CFG["latent"], img=GIMG):
        super().__init__()
        self.emb = nn.Embedding(nc, latent); self.init = img // 8
        self.fc = nn.Linear(latent, 256 * self.init ** 2)
        self.net = nn.Sequential(
            nn.BatchNorm2d(256), nn.Upsample(scale_factor=2),
            nn.Conv2d(256, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(True), nn.Upsample(scale_factor=2),
            nn.Conv2d(128, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(True), nn.Upsample(scale_factor=2),
            nn.Conv2d(64, ch, 3, 1, 1), nn.Tanh())
    def forward(self, z, y):
        return self.net(self.fc(z * self.emb(y)).view(z.size(0), 256, self.init, self.init))

class Critic(nn.Module):
    def __init__(self, nc, ch, img=GIMG):
        super().__init__()
        self.img = img; self.emb = nn.Embedding(nc, img * img)
        def blk(i, o, norm=True):
            L = [nn.Conv2d(i, o, 3, 2, 1)]
            if norm: L.append(nn.InstanceNorm2d(o, affine=True))
            L.append(nn.LeakyReLU(0.2, True)); return L
        self.net = nn.Sequential(*blk(ch + 1, 16, False), *blk(16, 32), *blk(32, 64), *blk(64, 128))
        self.out = nn.Linear(128 * (img // 16) ** 2, 1)
    def forward(self, x, y):
        m = self.emb(y).view(y.size(0), 1, self.img, self.img)
        return self.out(self.net(torch.cat([x, m], 1)).flatten(1))

def diffaug(x, ratio=0.125, cut=0.5):
    """Differentiable augmentation (translation + cutout), Zhao et al. 2020, applied to the critic input only."""
    B, C, H, W = x.shape; dv = x.device
    sh = int(H * ratio + 0.5)
    tx = torch.randint(-sh, sh + 1, (B, 1, 1), device=dv); ty = torch.randint(-sh, sh + 1, (B, 1, 1), device=dv)
    gb, gx, gy = torch.meshgrid(torch.arange(B, device=dv), torch.arange(H, device=dv), torch.arange(W, device=dv), indexing="ij")
    gx = torch.clamp(gx + tx + 1, 0, H + 1); gy = torch.clamp(gy + ty + 1, 0, W + 1)
    x = F.pad(x, [1, 1, 1, 1]).permute(0, 2, 3, 1).contiguous()[gb, gx, gy].permute(0, 3, 1, 2)
    cs = int(H * cut + 0.5)
    ox = torch.randint(0, H + (1 - cs % 2), (B, 1, 1), device=dv); oy = torch.randint(0, W + (1 - cs % 2), (B, 1, 1), device=dv)
    gb, gx, gy = torch.meshgrid(torch.arange(B, device=dv), torch.arange(cs, device=dv), torch.arange(cs, device=dv), indexing="ij")
    gx = torch.clamp(gx + ox - cs // 2, 0, H - 1); gy = torch.clamp(gy + oy - cs // 2, 0, W - 1)
    mask = torch.ones(B, H, W, device=dv); mask[gb, gx, gy] = 0
    return x * mask.unsqueeze(1)

def grad_penalty(C, real, fake, y):
    a = torch.rand(real.size(0), 1, 1, 1, device=real.device)
    x = (a * real + (1 - a) * fake).requires_grad_(True)
    g = torch.autograd.grad(C(x, y).sum(), x, create_graph=True)[0]
    return ((g.flatten(1).norm(2, dim=1) - 1) ** 2).mean()

def gan_paths(variant, seed, regime):
    stem = f"{variant}_s{seed}_r{regime}"
    return GAN_DIR / f"{stem}.pt", GAN_DIR / f"{stem}_state.pt", GAN_DIR / f"{stem}_hist.json"

def train_gan(variant, seed, regime, X, Y):
    """X: float (N,CH,GIMG,GIMG) in [-1,1] on device. Fixed number of critic iterations, resumable from Drive."""
    final, state, histp = gan_paths(variant, seed, regime)
    use_da = variant.endswith("_da")
    seed_everything(seed * 1000 + regime)
    G, C = Generator(NC, CH).to(DEVICE), Critic(NC, CH).to(DEVICE)
    oG = torch.optim.Adam(G.parameters(), lr=CFG["gan_lr"], betas=(0.0, 0.9))
    oC = torch.optim.Adam(C.parameters(), lr=CFG["gan_lr"], betas=(0.0, 0.9))
    hist = dict(it=[], c_loss=[], w_dist=[], g_loss=[]); it0 = 0
    if state.exists():
        ck = torch.load(state, map_location=DEVICE, weights_only=False)
        G.load_state_dict(ck["G"]); C.load_state_dict(ck["C"]); oG.load_state_dict(ck["oG"]); oC.load_state_dict(ck["oC"])
        hist, it0 = ck["hist"], ck["it"]; log(f"[GAN {DS} {variant} s{seed} r{regime}] resuming at iteration {it0}")
    N = X.size(0); bs = min(CFG["gan_bs"], N); nc_ = CFG["n_critic"]; lam = CFG["lambda_gp"]
    li = min(100, CFG["gan_iters"]); rc, rw, rg = [], [], []; t0 = time.time()
    for it in range(it0, CFG["gan_iters"]):
        idx = torch.randint(0, N, (bs,), device=DEVICE); real, y = X[idx], Y[idx]
        fake = G(torch.randn(bs, CFG["latent"], device=DEVICE), y)
        rA, fA = (diffaug(real), diffaug(fake.detach())) if use_da else (real, fake.detach())
        cr, cf = C(rA, y).mean(), C(fA, y).mean()
        lc = cf - cr + lam * grad_penalty(C, rA, fA, y)
        oC.zero_grad(set_to_none=True); lc.backward(); oC.step()
        rc.append(lc.detach()); rw.append((cr - cf).detach())
        if (it + 1) % nc_ == 0:
            fake = G(torch.randn(bs, CFG["latent"], device=DEVICE), y)
            lg = -C(diffaug(fake) if use_da else fake, y).mean()
            oG.zero_grad(set_to_none=True); lg.backward(); oG.step(); rg.append(lg.detach())
        if (it + 1) % li == 0:
            hist["it"].append(it + 1); hist["c_loss"].append(torch.stack(rc).mean().item())
            hist["w_dist"].append(torch.stack(rw).mean().item())
            hist["g_loss"].append(torch.stack(rg).mean().item() if rg else float("nan")); rc, rw, rg = [], [], []
        if (it + 1) % CFG["gan_ckpt_every"] == 0 and (it + 1) < CFG["gan_iters"]:
            atomic_torch_save(dict(G=G.state_dict(), C=C.state_dict(), oG=oG.state_dict(), oC=oC.state_dict(), hist=hist, it=it + 1), state)
    atomic_torch_save(G.state_dict(), final); save_json(hist, histp)
    if state.exists(): state.unlink()
    log(f"[GAN {DS} {variant} s{seed} r{regime}] done in {time.time() - t0:.0f}s | last W-dist {hist['w_dist'][-1]:.3f}")
    return G

def get_generator(variant, seed, regime, X, Y):
    final, _, _ = gan_paths(variant, seed, regime)
    if final.exists():
        G = Generator(NC, CH).to(DEVICE); G.load_state_dict(torch.load(final, map_location=DEVICE))
    else:
        G = train_gan(variant, seed, regime, X, Y)
    G.eval(); return G

@torch.no_grad()
def generate(G, cls, n, seed):
    G.eval(); g = torch.Generator(device=DEVICE).manual_seed(seed * 100_003 + cls * 17)
    out = []
    for i in range(0, n, 256):
        m = min(256, n - i)
        z = torch.randn(m, CFG["latent"], device=DEVICE, generator=g)
        out.append(G(z, torch.full((m,), cls, dtype=torch.long, device=DEVICE)))
    return torch.cat(out)

def show_img(ax, t01):
    a = t01.cpu().numpy()
    ax.imshow(a[0] if a.shape[0] == 1 else a.transpose(1, 2, 0), cmap="gray" if a.shape[0] == 1 else None, vmin=0, vmax=1); ax.axis("off")

def save_grid(G, path, seed, n=8, max_rows=8):
    rows = min(NC, max_rows)
    fig, ax = plt.subplots(rows, n, figsize=(n * 1.1, rows * 1.15))
    for c in range(rows):
        s = generate(G, c, n, seed + 9)
        for j in range(n): show_img(ax[c, j], (s[j] * 0.5 + 0.5).clamp(0, 1))
        ax[c, 0].set_title(CLASSES[c][:28], fontsize=6, loc="left")
    plt.tight_layout(); fig.savefig(path, dpi=130); plt.close(fig)

## 6. Classifiers (ResNet-18, DenseNet-121) and evaluation

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)
AUG = v2.Compose([v2.RandomRotation(15), v2.RandomHorizontalFlip(),
                  v2.RandomResizedCrop((IMG, IMG), scale=(0.8, 1.0), antialias=True),
                  v2.ColorJitter(brightness=0.2, contrast=0.2)])

def prepare(xu8, aug):
    # Augmentation must run in full precision: torchvision's rotation returns garbage inside autocast (fp16 grid maths).
    with torch.autocast(device_type=DEVICE.type, enabled=False):
        x = xu8.float().div(255.0)
        if x.size(1) == 1: x = x.expand(-1, 3, -1, -1)
        if aug: x = torch.stack([AUG(xi).clamp(0, 1) for xi in x])
        return (x - MEAN) / STD

def new_model(clf, pretrained):
    if clf == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        m.fc = nn.Linear(m.fc.in_features, NC)
    elif clf == "densenet121":
        m = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None)
        m.classifier = nn.Linear(m.classifier.in_features, NC)
    else:
        raise ValueError(clf)
    return m

_INIT = {}
def init_state(clf, seed):
    key = (clf, seed, NC)
    if key not in _INIT:
        torch.manual_seed(seed)                       # fixes the new output layer; the backbone is pretrained
        _INIT[key] = {k: v.clone() for k, v in new_model(clf, CFG["pretrained"]).state_dict().items()}
    return _INIT[key]

def autocast_ctx(enabled): return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=enabled)
def make_scaler(enabled):
    try: return torch.amp.GradScaler("cuda", enabled=enabled)
    except Exception: return torch.cuda.amp.GradScaler(enabled=enabled)

@torch.no_grad()
def predict_probs(model, X, use_amp):
    model.eval(); out = []
    for i in range(0, X.size(0), 128):
        with autocast_ctx(use_amp): lo = model(prepare(X[i:i + 128], False))
        out.append(F.softmax(lo.float(), 1))
    return torch.cat(out).cpu().numpy().astype(np.float64)

def compute_metrics(y, probs):
    probs = probs / probs.sum(1, keepdims=True); pred = probs.argmax(1); labs = list(range(NC))
    p, r, f1, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    pcp, pcr, pcf, pcs = precision_recall_fscore_support(y, pred, average=None, labels=labs, zero_division=0)
    return dict(
        accuracy=accuracy_score(y, pred), macro_precision=p, macro_recall=r, macro_f1=f1,
        balanced_accuracy=balanced_accuracy_score(y, pred),
        auc_roc=roc_auc_score(y, probs, multi_class="ovr", average="macro", labels=labs),
        mcc=matthews_corrcoef(y, pred), kappa=cohen_kappa_score(y, pred),
        per_class={CLASSES[i]: dict(precision=pcp[i], recall=pcr[i], f1=pcf[i], support=int(pcs[i])) for i in range(NC)},
        cm=confusion_matrix(y, pred, labels=labs).tolist()), pred

def run_classifier(clf, seed, X, y, aug, steps, test_X):
    use_amp = CFG["amp"] and DEVICE.type == "cuda"
    model = new_model(clf, False); model.load_state_dict(init_state(clf, seed)); model.to(DEVICE).train()
    opt = torch.optim.Adam(model.parameters(), lr=CFG["clf_lr"]); scaler = make_scaler(use_amp)
    gen = torch.Generator().manual_seed(seed * 7919 + steps)
    bs, N, step, losses = CFG["clf_bs"], X.size(0), 0, []
    t0 = time.time()
    while step < steps:
        perm = torch.randperm(N, generator=gen).to(DEVICE)
        for i in range(0, N - bs + 1, bs):
            if step >= steps: break
            idx = perm[i:i + bs]
            xb = prepare(X[idx], aug)                                   # augmentation outside autocast
            with autocast_ctx(use_amp): loss = F.cross_entropy(model(xb), y[idx])
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            losses.append(loss.detach()); step += 1
    last = torch.stack(losses[-max(1, steps // CFG["clf_epochs_base"]):]).mean().item()
    probs = predict_probs(model, test_X, use_amp)
    metrics, pred = compute_metrics(TEST["y"], probs)
    metrics.update(train_loss_last_epoch=last, train_time_s=time.time() - t0)
    del model, opt
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return metrics, pred

## 7. Experiment runner (resumable; one file per finished run, saved straight to Drive)

In [ ]:
def build_arms():
    arms = [dict(name="baseline", cond="baseline", variant=None, ratio=None, aug=False),
            dict(name="classical", cond="classical", variant=None, ratio=None, aug=True)]
    for v in CFG["gan_variants"]:
        for r in CFG["ratios"]:
            arms.append(dict(name=arm_name(v, r), cond="gan", variant=v, ratio=r, aug=False))
        for r in CFG["combo_ratios"]:
            arms.append(dict(name=arm_name(v, r, True), cond="gan+classical", variant=v, ratio=r, aug=True))
    return arms
ARMS = build_arms()

def run_path(ds, clf, pipe, arm_nm, regime, seed):
    return ds_dirs(ds)[1] / f"{clf}__{pipe}__{arm_nm}__r{regime}__s{seed}.json"

def build_jobs(datasets):
    jobs = []
    for ds in datasets:
        for clf, plan in CFG["clf_plan"].items():
            seeds = plan["seeds"] if plan["seeds"] is not None else CFG["ds_seeds"][ds]
            for seed in seeds:
                for regime in plan["regimes"]:
                    for pipe in CFG["pipelines"]:
                        for a in ARMS: jobs.append((ds, clf, seed, regime, pipe, a))
    return jobs

def pending_jobs(datasets):
    return [j for j in build_jobs(datasets) if not run_path(j[0], j[1], j[4], j[5]["name"], j[3], j[2]).exists()]

def per_class_n(n, ratio): return max(1, int(round(n * ratio)))
T_START = time.time()
def out_of_time():
    return MAX_RUNTIME_MIN is not None and (time.time() - T_START) / 60 > MAX_RUNTIME_MIN

def execute(datasets):
    todo_all = pending_jobs(datasets); total = len(todo_all); done = 0; t0 = time.time()
    log(f"{total} runs pending for {datasets} of {len(build_jobs(datasets))} planned")
    by_ds = {}
    for j in todo_all: by_ds.setdefault(j[0], []).append(j)
    for ds, jobs in by_ds.items():
        D = get_dataset(ds); use_dataset(D)
        groups = {}
        for (_, clf, seed, regime, pipe, a) in jobs: groups.setdefault((seed, regime), []).append((clf, pipe, a))
        subs_cache = {}
        for (seed, regime) in sorted(groups):
            if out_of_time(): log("MAX_RUNTIME_MIN reached, stopping cleanly. Run again to continue."); return
            if regime > D["report"]["max_regime"]:
                log(f"[{ds}] skipping regime {regime}: smallest class has only {D['report']['max_regime']} usable images"); continue
            if seed not in subs_cache: subs_cache[seed] = make_subsets(seed)
            idx = subs_cache[seed][regime]; yreal = torch.as_tensor(POOL["y"][idx], device=DEVICE)
            todo = groups[(seed, regime)]; synth = {}
            for v in sorted({a["variant"] for _, _, a in todo if a["variant"]}):
                X64 = torch.as_tensor(POOL["x64"][idx], device=DEVICE).float().div(127.5).sub(1)
                G = get_generator(v, seed, regime, X64, yreal)
                gp = FIG_DIR / "gan_samples" / ds; gp.mkdir(parents=True, exist_ok=True)
                if not (gp / f"{v}_s{seed}_r{regime}.png").exists(): save_grid(G, gp / f"{v}_s{seed}_r{regime}.png", seed)
                per_max = per_class_n(regime, max(a["ratio"] for _, _, a in todo if a["variant"] == v))
                synth[v] = [synth_to_img(generate(G, c, per_max, seed)) for c in range(NC)]
                del G
            for pipe in CFG["pipelines"]:
                Xr = real_images(idx, pipe)
                for clf, p, arm in todo:
                    if p != pipe: continue
                    if out_of_time(): log("MAX_RUNTIME_MIN reached, stopping cleanly. Run again to continue."); return
                    n_syn = 0; X, y = Xr, yreal
                    if arm["variant"]:
                        k = per_class_n(regime, arm["ratio"]); n_syn = k * NC
                        X = torch.cat([Xr] + [synth[arm["variant"]][c][:k] for c in range(NC)])
                        y = torch.cat([yreal] + [torch.full((k,), c, dtype=torch.long, device=DEVICE) for c in range(NC)])
                    n_real = len(idx)
                    steps = max(1, CFG["clf_epochs_base"] * ((n_real if CFG["equal_steps"] else X.size(0)) // CFG["clf_bs"]))
                    m, pred = run_classifier(clf, seed, X, y, arm["aug"], steps, TEST_X[pipe])
                    rp = run_path(ds, clf, pipe, arm["name"], regime, seed)
                    rec = dict(tag=rp.stem, dataset=ds, clf=clf, pipeline=pipe, arm=arm["name"], cond=arm["cond"],
                               variant=arm["variant"], ratio=arm["ratio"], regime=regime, seed=seed, n_real=n_real,
                               n_synth=n_syn, steps=steps, phash=PHASH, finished=time.strftime("%Y-%m-%d %H:%M:%S"), **m)
                    np.savez_compressed(rp.with_suffix(".npz"), preds=pred.astype(np.int8))
                    save_json(rec, rp)                                  # JSON last: its existence marks completion
                    done += 1; el = time.time() - t0
                    log(f"[{done}/{total}] {rec['tag']} acc={m['accuracy']:.4f} f1={m['macro_f1']:.4f} "
                        f"| {el / 60:.1f} min elapsed, ETA {el / done * (total - done) / 60:.1f} min")
                del Xr
            del synth
    left = len(pending_jobs(datasets)); log("all planned runs complete" if left == 0 else f"{left} runs still pending (see skip messages above)")

### 7.1 Plan size and progress (safe to run any time; nothing is trained here)

In [ ]:
def progress(datasets=None):
    datasets = datasets or CFG["datasets"]; rows = []
    for ds in datasets:
        for clf, plan in CFG["clf_plan"].items():
            seeds = plan["seeds"] if plan["seeds"] is not None else CFG["ds_seeds"][ds]
            planned = len(seeds) * len(plan["regimes"]) * len(CFG["pipelines"]) * len(ARMS)
            done = len(list(ds_dirs(ds)[1].glob(f"{clf}__*.json")))
            gans = len(list(ds_dirs(ds)[0].glob("*_hist.json")))
            rows.append(dict(dataset=ds, classifier=clf, planned_runs=planned, finished=min(done, planned) if done <= planned else done, GANs_trained=gans))
    df = pd.DataFrame(rows); df["left"] = (df.planned_runs - df.finished).clip(lower=0); display(df); return df
prog = progress()
print("Total classifier runs planned:", int(prog.planned_runs.sum()), "| left:", int(prog["left"].sum()))
print("Rough guide: the pilot is the cheapest way to measure real speed on your GPU; check the ETA in the log, then decide whether to trim clf_plan or seeds.")

In [ ]:
execute(RUN_DATASETS)

## 8. Aggregate results (reads the saved run files from Drive)

In [ ]:
def load_runs(datasets):
    rows, pcs = [], []
    for ds in datasets:
        for p in sorted(ds_dirs(ds)[1].glob("*.json")):
            r = json.load(open(p))
            rows.append({k: v for k, v in r.items() if k not in ("per_class", "cm")})
            for cls, m in r["per_class"].items():
                pcs.append({**{k: r[k] for k in ("dataset", "clf", "pipeline", "arm", "cond", "variant", "ratio", "regime", "seed")}, "class": cls, **m})
    return pd.DataFrame(rows), pd.DataFrame(pcs)

runs, pcdf = load_runs(CFG["datasets"])
assert len(runs), "No finished runs found yet. Run the runner cell first."
runs.to_csv(TAB_DIR / "all_runs.csv", index=False); pcdf.to_csv(TAB_DIR / "per_class_runs.csv", index=False)
GK = ["dataset", "clf", "pipeline", "regime"]
cnt = runs.groupby(GK + ["arm"]).size().unstack("arm")
print("runs loaded:", len(runs)); display(cnt)
_left = int(progress(CFG["datasets"])["left"].sum())
print("All planned runs are finished." if _left == 0 else f"NOTE: {_left} planned runs are still pending (see the progress table). Tables below cover finished runs only.")

### 8.1 Summary tables and paired comparisons

In [ ]:
METRICS = ["accuracy", "macro_f1", "balanced_accuracy", "auc_roc"]
def summarize(runs, metric):
    s = runs.groupby(GK + ["arm"])[metric].agg(n="count", mean="mean", sd="std").reset_index()
    s["ci95"] = [stats.t.ppf(0.975, n - 1) * sd / np.sqrt(n) if n > 1 else np.nan for n, sd in zip(s.n, s.sd)]
    return s

def holm(p):
    p = np.asarray(p, float); out = np.full_like(p, np.nan); ok = ~np.isnan(p); m = int(ok.sum())
    if m == 0: return out
    order = np.argsort(p[ok]); adj = np.empty(m); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[ok][i]); adj[i] = min(1.0, run)
    out[ok] = adj; return out

def paired_table(runs, metric="accuracy", ref="baseline", n_boot=10000):
    rng = np.random.RandomState(0); rows = []
    for key, sub in runs.groupby(GK):
        piv = sub.pivot(index="seed", columns="arm", values=metric)
        if ref not in piv: continue
        for arm in piv.columns:
            if arm == ref: continue
            d = (piv[arm] - piv[ref]).dropna().values; n = len(d)
            if n < 2: continue
            boot = rng.choice(d, size=(n_boot, n), replace=True).mean(1)
            try: pval = stats.wilcoxon(d).pvalue
            except ValueError: pval = np.nan
            sd = d.std(ddof=1)
            rows.append(dict(zip(GK, key), arm=arm, n=n, mean_diff=d.mean(), ci_lo=np.percentile(boot, 2.5),
                             ci_hi=np.percentile(boot, 97.5), wilcoxon_p=pval, wins=int((d > 0).sum()),
                             cohen_dz=d.mean() / sd if sd > 0 else np.nan))
    out = pd.DataFrame(rows)
    if not out.empty:
        out["p_holm"] = out.groupby(["dataset", "clf", "pipeline"])["wilcoxon_p"].transform(lambda s: pd.Series(holm(s.values), index=s.index))
    return out

SUMM = {m: summarize(runs, m) for m in METRICS}
PAIR = {m: paired_table(runs, m) for m in ["accuracy", "macro_f1"]}
for m, df in SUMM.items(): df.to_csv(TAB_DIR / f"summary_{m}.csv", index=False)
for m, df in PAIR.items(): df.to_csv(TAB_DIR / f"paired_vs_baseline_{m}.csv", index=False)

s = SUMM["accuracy"]
for (ds, clf, pipe), t in s.groupby(["dataset", "clf", "pipeline"]):
    t = t.copy(); t["val"] = t.apply(lambda r: f"{r['mean']:.3f} +/- {r['ci95']:.3f} (n={r['n']})" if r["n"] > 1 else f"{r['mean']:.3f} (n=1)", axis=1)
    print(f"\n=== accuracy | {ds} | {clf} | {pipe} (mean, 95% CI over seeds) ===")
    display(t.pivot(index="arm", columns="regime", values="val").sort_index(axis=1, ascending=False))
print("\nPaired differences vs baseline (accuracy). The Wilcoxon p-value cannot go below 2/2^n for n seeds, so lean on the bootstrap CI.")
display(PAIR["accuracy"].round(4))

## 9. Figures (saved to Drive as PNG, 300 dpi)

In [ ]:
C_ARM = {"baseline": "#2a78d6", "classical": "#eb6834", "gan": "#1baf7a", "gan+classical": "#6250d6"}
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e6e5e1"
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 10, "axes.edgecolor": INK2,
                     "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2, "text.color": INK,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True})
LABEL = {"baseline": "No augmentation", "classical": "Classical", PRIMARY: f"WGAN-GP {int(CFG['primary_ratio'] * 100)}%",
         PRIMARY_COMBO: f"WGAN-GP {int(CFG['primary_ratio'] * 100)}% + classical"}
PIPE_TITLE = {"native": "Native (real sharp, synthetic upsampled 64px)", "matched64": "Matched (all images 64px upsampled)"}
DS_TITLE = {"brain_mri": "Brain tumor MRI", "plantvillage": "PlantVillage (15 classes)"}
def arm_color(a): return C_ARM["baseline"] if a == "baseline" else C_ARM["classical"] if a == "classical" else C_ARM["gan+classical"] if a.endswith("+cls") else C_ARM["gan"]
def savefig(fig, name): fig.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight"); plt.show()
def reg_order(df): return sorted(df.regime.unique(), reverse=True)

def fig_main(ds, clf, metric="accuracy"):
    s = SUMM[metric]; s = s[(s.dataset == ds) & (s.clf == clf)]
    if s.empty: return
    REG = reg_order(s); arms = [a for a in ["baseline", "classical", PRIMARY, PRIMARY_COMBO] if a in set(s.arm)]
    pipes = [p for p in CFG["pipelines"] if p in set(s.pipeline)]
    fig, axes = plt.subplots(1, len(pipes), figsize=(5.6 * len(pipes), 4.2), sharey=True); axes = np.atleast_1d(axes)
    for ax, pipe in zip(axes, pipes):
        for a in arms:
            t = s[(s.pipeline == pipe) & (s.arm == a)].set_index("regime").reindex(REG)
            ax.errorbar(range(len(REG)), t["mean"], yerr=t["ci95"].fillna(0), color=arm_color(a), marker="o", ms=6, lw=2, capsize=3, label=LABEL.get(a, a))
        ax.set_xticks(range(len(REG))); ax.set_xticklabels(REG); ax.set_xlabel("Real training images per class")
        ax.set_title(PIPE_TITLE[pipe], fontsize=9, loc="left")
    axes[0].set_ylabel(metric.replace("_", " ").title() + " (mean, 95% CI)"); axes[0].legend(frameon=False, fontsize=8)
    fig.suptitle(f"{DS_TITLE.get(ds, ds)} | {clf}", x=0.01, ha="left", fontsize=10); plt.tight_layout(); savefig(fig, f"fig1_main_{ds}_{clf}_{metric}")

def fig_overview(clf, metric="accuracy"):
    """Headline figure: paired improvement over baseline, all datasets, both pipelines."""
    P = PAIR[metric]; P = P[P.clf == clf]
    if P.empty: return
    dss = [d for d in CFG["datasets"] if d in set(P.dataset)]; pipes = [p for p in CFG["pipelines"] if p in set(P.pipeline)]
    arms = [a for a in ["classical", PRIMARY] if a in set(P.arm)]
    fig, axes = plt.subplots(len(dss), len(pipes), figsize=(5.2 * len(pipes), 3.4 * len(dss)), sharey=True, squeeze=False)
    for i, ds in enumerate(dss):
        for j, pipe in enumerate(pipes):
            ax = axes[i, j]; REG = reg_order(P[P.dataset == ds])
            for k, a in enumerate(arms):
                t = P[(P.dataset == ds) & (P.pipeline == pipe) & (P.arm == a)].set_index("regime").reindex(REG)
                x = np.arange(len(REG)) + (k - 0.5) * 0.18
                ax.errorbar(x, t.mean_diff, yerr=[(t.mean_diff - t.ci_lo).fillna(0), (t.ci_hi - t.mean_diff).fillna(0)], fmt="o", ms=7, capsize=3, color=arm_color(a), label=LABEL.get(a, a))
            ax.axhline(0, color=INK2, lw=1); ax.set_xticks(range(len(REG))); ax.set_xticklabels(REG)
            ax.set_title(f"{DS_TITLE.get(ds, ds)} | {pipe}", fontsize=9, loc="left")
            if i == len(dss) - 1: ax.set_xlabel("Real training images per class")
            if j == 0: ax.set_ylabel(f"Paired {metric.replace('_', ' ')} gain vs baseline")
    axes[0, 0].legend(frameon=False, fontsize=8); fig.suptitle(f"{clf}: gain over no augmentation (95% bootstrap CI)", x=0.01, ha="left", fontsize=10)
    plt.tight_layout(); savefig(fig, f"fig2_overview_gain_{clf}_{metric}")

def fig_ratio(ds, clf, metric="accuracy"):
    s = SUMM[metric]; s = s[(s.dataset == ds) & (s.clf == clf)]
    if s.empty: return
    REG = reg_order(s); pipes = [p for p in CFG["pipelines"] if p in set(s.pipeline)]; v = PRIMARY_VARIANT; shades = ["#99dcc4", "#5cc7a1", "#1baf7a", "#0b6a4a"]
    fig, axes = plt.subplots(1, len(pipes), figsize=(5.6 * len(pipes), 4.0), sharey=True); axes = np.atleast_1d(axes)
    for ax, pipe in zip(axes, pipes):
        for k, reg in enumerate(REG):
            rows = [(r * 100, s[(s.pipeline == pipe) & (s.regime == reg) & (s.arm == arm_name(v, r))]) for r in CFG["ratios"]]
            xs = [x for x, t in rows if len(t)]; ys = [t["mean"].iloc[0] for _, t in rows if len(t)]; es = [t["ci95"].iloc[0] for _, t in rows if len(t)]
            ax.errorbar(xs, ys, yerr=np.nan_to_num(es), color=shades[k % 4], marker="o", ms=6, lw=2, capsize=3, label=f"{reg}/class")
            b = s[(s.pipeline == pipe) & (s.regime == reg) & (s.arm == "baseline")]
            if len(b): ax.axhline(b["mean"].iloc[0], color=shades[k % 4], lw=1, ls="--")
        ax.set_xlabel("Synthetic images added (% of real)"); ax.set_title(PIPE_TITLE[pipe], fontsize=9, loc="left")
    axes[0].set_ylabel(metric.replace("_", " ").title()); axes[0].legend(frameon=False, fontsize=8, title="Dashed = baseline")
    fig.suptitle(f"{DS_TITLE.get(ds, ds)} | {clf}: ratio sweep (reported separately, not used for the headline)", x=0.01, ha="left", fontsize=9)
    plt.tight_layout(); savefig(fig, f"fig3_ratio_sweep_{ds}_{clf}_{metric}")

def fig_perclass(ds, clf):
    d = pcdf[(pcdf.dataset == ds) & (pcdf.clf == clf)]
    if d.empty or d["class"].nunique() > 6: return                  # per-class bars only for the 4-class dataset
    arms = [a for a in ["baseline", "classical", PRIMARY] if a in set(d.arm)]
    for pipe in [p for p in CFG["pipelines"] if p in set(d.pipeline)]:
        REG = reg_order(d); fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharey=True)
        for ax, reg in zip(axes.flat, REG[:4]):
            sub = d[(d.pipeline == pipe) & (d.regime == reg)]; cl = list(dict.fromkeys(sub["class"]))
            for k, a in enumerate(arms):
                g = sub[sub.arm == a].groupby("class")["recall"].agg(["mean", "std"]).reindex(cl)
                ax.bar(np.arange(len(cl)) + (k - 1) * 0.26, g["mean"], 0.24, yerr=g["std"].fillna(0), color=arm_color(a), label=LABEL.get(a, a), capsize=2)
            ax.set_xticks(range(len(cl))); ax.set_xticklabels(cl, fontsize=8); ax.set_title(f"{reg} real images/class", fontsize=9, loc="left")
        axes[0, 0].legend(frameon=False, fontsize=8); fig.suptitle(f"Per-class recall | {ds} | {clf} | {pipe}", x=0.01, ha="left", fontsize=10)
        plt.tight_layout(); savefig(fig, f"fig4_perclass_recall_{ds}_{clf}_{pipe}")

def fig_confusion(ds, clf, pipe, regime):
    arms = [a for a in ["baseline", "classical", PRIMARY] if len(list(ds_dirs(ds)[1].glob(f"{clf}__{pipe}__{a}__r{regime}__s*.json")))]
    if not arms: return
    fig, axes = plt.subplots(1, len(arms), figsize=(4.2 * len(arms), 3.9)); axes = np.atleast_1d(axes)
    for ax, a in zip(axes, arms):
        cm = None
        for p in ds_dirs(ds)[1].glob(f"{clf}__{pipe}__{a}__r{regime}__s*.json"):
            m = np.array(json.load(open(p))["cm"], float); cm = m if cm is None else cm + m
        cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1); n = len(cmn)
        ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1); ax.grid(False)
        if n <= 6:
            for i in range(n):
                for j in range(n): ax.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", fontsize=8, color="white" if cmn[i, j] > 0.5 else INK)
            names = [c[:12] for c in (json.load(open(next(iter(ds_dirs(ds)[1].glob(f"{clf}__{pipe}__{a}__r{regime}__s*.json")))))["per_class"].keys())]
            ax.set_xticks(range(n)); ax.set_yticks(range(n)); ax.set_xticklabels(names, rotation=35, ha="right", fontsize=7); ax.set_yticklabels(names, fontsize=7)
        ax.set_title(LABEL.get(a, a), fontsize=9, loc="left"); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    fig.suptitle(f"Row-normalised confusion, all seeds | {ds} | {clf} | {pipe} | {regime}/class", x=0.01, ha="left", fontsize=9)
    plt.tight_layout(); savefig(fig, f"fig5_confusion_{ds}_{clf}_{pipe}_r{regime}")

for ds in CFG["datasets"]:
    for clf in CFG["clf_plan"]:
        fig_main(ds, clf); fig_ratio(ds, clf); fig_perclass(ds, clf)
        for pipe in CFG["pipelines"]:
            r = runs[(runs.dataset == ds) & (runs.clf == clf)]
            if len(r): fig_confusion(ds, clf, pipe, int(r.regime.min()))
for clf in CFG["clf_plan"]: fig_overview(clf)
fig_main_done = True

### 9.1 GAN diagnostics: training curves and pipeline illustration (loads datasets, needs trained GANs)

In [ ]:
def fig_gan_curves(ds):
    gdir = ds_dirs(ds)[0]; hs = sorted(gdir.glob(f"{PRIMARY_VARIANT}_s*_r*_hist.json"))
    if not hs: return
    seed0 = min(int(p.stem.split("_s")[1].split("_")[0]) for p in hs); shades = ["#99dcc4", "#5cc7a1", "#1baf7a", "#0b6a4a"]
    fig, ax = plt.subplots(figsize=(6, 3.8)); regs = sorted({int(p.stem.split("_r")[1].split("_")[0]) for p in hs}, reverse=True)
    for k, reg in enumerate(regs):
        hp = gdir / f"{PRIMARY_VARIANT}_s{seed0}_r{reg}_hist.json"
        if hp.exists(): h = json.load(open(hp)); ax.plot(h["it"], h["w_dist"], color=shades[k % 4], lw=1.8, label=f"{reg}/class")
    ax.set_xlabel("Critic iteration"); ax.set_ylabel("Estimated Wasserstein distance"); ax.legend(frameon=False, fontsize=8)
    ax.set_title(f"{DS_TITLE.get(ds, ds)} | seed {seed0}", fontsize=9, loc="left"); plt.tight_layout(); savefig(fig, f"fig6_gan_wdist_{ds}")

def fig_pipeline_illustration(ds):
    D = get_dataset(ds); use_dataset(D); seeds = CFG["ds_seeds"][ds]; seed = seeds[0]
    regs = [r for r in ALL_REGIMES if gan_paths(PRIMARY_VARIANT, seed, r)[0].exists()]
    if not regs: return
    regime = regs[0]; idx = make_subsets(seed)[regime]
    G = Generator(NC, CH).to(DEVICE); G.load_state_dict(torch.load(gan_paths(PRIMARY_VARIANT, seed, regime)[0], map_location=DEVICE)); G.eval()
    native, matched = real_images(idx, "native").cpu(), real_images(idx, "matched64").cpu(); rows = min(NC, 5)
    fig, ax = plt.subplots(rows, 3, figsize=(5.4, rows * 1.8))
    for c in range(rows):
        j = int(np.where(POOL["y"][idx] == c)[0][0]); s = synth_to_img(generate(G, c, 1, seed + 5)).cpu()[0]
        for k, (im, t) in enumerate([(native[j], "Real, native"), (matched[j], "Real, matched64"), (s, "Synthetic")]):
            show_img(ax[c, k], im.float() / 255)
            if c == 0: ax[c, k].set_title(t, fontsize=8)
        ax[c, 0].text(-0.05, 0.5, CLASSES[c][:16], transform=ax[c, 0].transAxes, ha="right", va="center", fontsize=7)
    plt.tight_layout(); savefig(fig, f"fig7_pipeline_illustration_{ds}")

for ds in CFG["datasets"]:
    if ds in RUN_DATASETS or ds in DATASETS:
        fig_gan_curves(ds)
        try: fig_pipeline_illustration(ds)
        except Exception as e: print("illustration skipped:", e)

## 10. Generator quality: KID against a real-vs-real floor

In [ ]:
def _prep299(a64):
    x = torch.as_tensor(np.asarray(a64), device=DEVICE).float().div(255)
    if x.size(1) == 1: x = x.repeat(1, 3, 1, 1)
    return F.interpolate(x, size=(299, 299), mode="bilinear", align_corners=False).clamp(0, 1)

def kid_pair(a, b):
    from torchmetrics.image.kid import KernelInceptionDistance
    ss = min(50, len(a), len(b))
    kid = KernelInceptionDistance(feature=2048, subset_size=ss, subsets=100, normalize=True).to(DEVICE)
    for i in range(0, len(a), 64): kid.update(_prep299(a[i:i + 64]), real=True)
    for i in range(0, len(b), 64): kid.update(_prep299(b[i:i + 64]), real=False)
    m, s_ = kid.compute(); return float(m), float(s_)

def run_kid(datasets):
    if SMOKE: print("KID skipped in smoke test"); return pd.DataFrame()
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torchmetrics", "torch-fidelity"]); import torchmetrics  # noqa
    except Exception as e:
        print("Could not set up torchmetrics, skipping KID:", e); return pd.DataFrame()
    rows = []
    for ds in datasets:
        D = get_dataset(ds); use_dataset(D)
        for v in CFG["gan_variants"]:
            for seed in CFG["ds_seeds"][ds]:
                sub = make_subsets(seed)
                for reg in sorted(sub, reverse=True):
                    f = KID_DIR / f"{v}_s{seed}_r{reg}.json"
                    if not f.exists():
                        gpath = gan_paths(v, seed, reg)[0]
                        if not gpath.exists(): continue
                        G = Generator(NC, CH).to(DEVICE); G.load_state_dict(torch.load(gpath, map_location=DEVICE)); G.eval()
                        idx = sub[reg]; real = POOL["x64"][idx]; n = len(idx)
                        rng = np.random.RandomState(500 + seed); others = np.setdiff1d(np.arange(len(POOL["y"])), idx)
                        other = POOL["x64"][rng.choice(others, size=n, replace=False)]
                        per = max(1, n // NC); synth = torch.cat([generate(G, c, per, seed + 3) for c in range(NC)])
                        s64 = ((synth * 0.5 + 0.5).clamp(0, 1) * 255).round().to(torch.uint8).cpu().numpy()
                        kr, _ = kid_pair(real, s64); kf, _ = kid_pair(real, other)
                        save_json(dict(dataset=ds, variant=v, seed=seed, regime=reg, kid_synth=kr, kid_real_floor=kf), f)
                    rows.append(json.load(open(f)))
    return pd.DataFrame(rows)

kid_df = run_kid([d for d in RUN_DATASETS])
if len(kid_df):
    kid_df.to_csv(TAB_DIR / "kid_scores.csv", index=False)
    for ds, kd in kid_df.groupby("dataset"):
        REG = sorted(kd.regime.unique(), reverse=True); g = kd.groupby("regime")[["kid_synth", "kid_real_floor"]].agg(["mean", "std"]).reindex(REG)
        display(g.round(5)); fig, ax = plt.subplots(figsize=(5.6, 3.8)); xs = np.arange(len(REG))
        ax.errorbar(xs, g[("kid_synth", "mean")], yerr=g[("kid_synth", "std")].fillna(0), color="#1baf7a", marker="o", ms=7, lw=2, capsize=3, label="Real vs synthetic")
        ax.errorbar(xs, g[("kid_real_floor", "mean")], yerr=g[("kid_real_floor", "std")].fillna(0), color="#52514e", marker="s", ms=7, lw=2, capsize=3, label="Real vs real (floor)")
        ax.set_xticks(xs); ax.set_xticklabels(REG); ax.set_xlabel("Real training images per class"); ax.set_ylabel("KID (lower is better)")
        ax.set_title(DS_TITLE.get(ds, ds), fontsize=9, loc="left"); ax.legend(frameon=False, fontsize=8); plt.tight_layout(); savefig(fig, f"fig8_kid_{ds}")

## 11. Headline numbers and download bundle

In [ ]:
def headline():
    out = {}
    for (ds, clf, pipe, reg), sub in runs.groupby(GK):
        row = {}
        for a in ["baseline", "classical", PRIMARY, PRIMARY_COMBO]:
            t = sub[sub.arm == a]
            if len(t): row[a] = dict(acc_mean=float(t.accuracy.mean()), acc_sd=float(t.accuracy.std()) if len(t) > 1 else None, f1_mean=float(t.macro_f1.mean()), n=int(len(t)))
        pr = PAIR["accuracy"]; pr = pr[(pr.dataset == ds) & (pr.clf == clf) & (pr.pipeline == pipe) & (pr.regime == reg)]
        row["paired_vs_baseline_accuracy"] = {r.arm: dict(diff=float(r.mean_diff), ci=[float(r.ci_lo), float(r.ci_hi)], p_holm=float(r.p_holm)) for r in pr.itertuples()}
        out.setdefault(ds, {}).setdefault(clf, {}).setdefault(pipe, {})[str(reg)] = row
    save_json(out, OUT / "headline_numbers.json"); return out

H = headline()
for ds in H:
    for clf in H[ds]:
        for pipe in H[ds][clf]:
            print(f"\n{ds} | {clf} | {pipe}")
            for reg in sorted(H[ds][clf][pipe], key=int, reverse=True):
                row = H[ds][clf][pipe][reg]
                print(f"  {reg:>4}/class  " + " | ".join(f"{LABEL.get(a, a)}: {v['acc_mean']:.3f}" for a, v in row.items() if isinstance(v, dict) and "acc_mean" in v))

bundle = OUT / "seminar_results_bundle.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in (FIG_DIR, TAB_DIR):
        for p in folder.rglob("*"):
            if p.is_file(): z.write(p, p.relative_to(OUT))
    for f in list(OUT.glob("data_report_*.json")) + [OUT / "headline_numbers.json", OUT / "env_and_config.json"]:
        if f.exists(): z.write(f, f.name)
print("\nBundle on Drive:", bundle)
if IN_COLAB:
    from google.colab import files
    files.download(str(bundle))

## 12. OPTIONAL: expert review of generated MRI (Fortis)
**Off by default.** Turn on with `RUN_FORTIS = True` only after the MRI GANs are trained. This builds a blinded image set (real held-out test images mixed with synthetic ones, both passed through the same 64px path so sharpness is no giveaway), an answer key, and a rating sheet for the reviewer. After the reviewer fills the sheet in, run `analyse_fortis(...)`.

This is a **realism review, not a diagnostic validation**. It asks whether an expert can tell real from synthetic and whether synthetic images look anatomically plausible. A single reviewer gives descriptive evidence only, so report it that way. Tell the reviewer what the images are for and get their agreement before you use their ratings in anything you publish.

In [ ]:
RUN_FORTIS = False
FORTIS_N_PER_CLASS = 10          # real and synthetic images per class
FORTIS_DIR = OUT / "fortis_review"

def build_fortis_set():
    D = get_dataset("brain_mri"); use_dataset(D)
    seed = CFG["ds_seeds"]["brain_mri"][0]; regs = [r for r in ALL_REGIMES if gan_paths(PRIMARY_VARIANT, seed, r)[0].exists()]
    assert regs, "Train the MRI GAN first (run section 7 for brain_mri)."
    reg = regs[0]; G = Generator(NC, CH).to(DEVICE); G.load_state_dict(torch.load(gan_paths(PRIMARY_VARIANT, seed, reg)[0], map_location=DEVICE)); G.eval()
    rng = np.random.RandomState(2026); items = []
    for c in range(NC):
        ti = np.where(TEST["y"] == c)[0]; pick = rng.choice(ti, size=min(FORTIS_N_PER_CLASS, len(ti)), replace=False)
        for i in pick: items.append(("real", CLASSES[c], TEST["x64"][i]))
        s = generate(G, c, FORTIS_N_PER_CLASS, seed + 77); s8 = ((s * 0.5 + 0.5).clamp(0, 1) * 255).round().to(torch.uint8).cpu().numpy()
        for a in s8: items.append(("synthetic", CLASSES[c], a))
    rng.shuffle(items)
    (FORTIS_DIR / "blinded_images").mkdir(parents=True, exist_ok=True)
    key, sheet = [], []
    for n, (truth, cls, a) in enumerate(items, 1):
        iid = f"img_{n:03d}"
        im = Image.fromarray(a[0]).resize((256, 256), Image.BILINEAR)           # same path for real and synthetic
        im.save(FORTIS_DIR / "blinded_images" / f"{iid}.png")
        key.append(dict(image_id=iid, truth=truth, label_class=cls))
        sheet.append(dict(image_id=iid, looks_real_or_synthetic="", realism_1to5="", class_guess="", plausible_anatomy_1to5="", comments=""))
    pd.DataFrame(key).to_csv(FORTIS_DIR / "ANSWER_KEY_do_not_share.csv", index=False)
    pd.DataFrame(sheet).to_csv(FORTIS_DIR / "rating_sheet.csv", index=False)
    per = 20; ids = [k["image_id"] for k in key]
    for pg in range(math.ceil(len(ids) / per)):
        fig, axes = plt.subplots(4, 5, figsize=(11, 9))
        for ax in axes.flat: ax.axis("off")
        for ax, iid in zip(axes.flat, ids[pg * per:(pg + 1) * per]):
            ax.imshow(Image.open(FORTIS_DIR / "blinded_images" / f"{iid}.png"), cmap="gray"); ax.set_title(iid, fontsize=8)
        plt.tight_layout(); fig.savefig(FORTIS_DIR / f"contact_sheet_{pg + 1:02d}.png", dpi=150); plt.close(fig)
    (FORTIS_DIR / "READ_ME_for_reviewer.txt").write_text(
        "Each image is a brain MRI slice, some real and some computer generated.\n"
        "For every image_id in rating_sheet.csv please fill in:\n"
        "  looks_real_or_synthetic : R or S\n  realism_1to5 : 1 = clearly artificial, 5 = indistinguishable from a real scan\n"
        "  class_guess : glioma, meningioma, notumor, pituitary or unsure\n"
        "  plausible_anatomy_1to5 : 1 = anatomically implausible, 5 = fully plausible\n  comments : optional\n"
        "Images are small (64px source) so please judge overall appearance. This is not a diagnostic task.\n")
    print("Reviewer pack written to", FORTIS_DIR, "| images:", len(items))

def analyse_fortis(filled_csv=None):
    filled_csv = filled_csv or (FORTIS_DIR / "rating_sheet_filled.csv")
    key = pd.read_csv(FORTIS_DIR / "ANSWER_KEY_do_not_share.csv"); r = pd.read_csv(filled_csv).merge(key, on="image_id")
    r["guess_correct"] = (r.looks_real_or_synthetic.str.upper().str[0] == r.truth.str[0].str.upper())
    print(f"Reviewer real-vs-synthetic accuracy: {r.guess_correct.mean():.3f} (chance = 0.50, n={len(r)})")
    print(r.groupby("truth")[["realism_1to5", "plausible_anatomy_1to5"]].agg(["mean", "std"]).round(2))
    a, b = r[r.truth == "real"].realism_1to5.dropna(), r[r.truth == "synthetic"].realism_1to5.dropna()
    if len(a) and len(b): print("Mann-Whitney U on realism (real vs synthetic): p =", round(stats.mannwhitneyu(a, b).pvalue, 4))
    r["class_ok"] = r.class_guess.str.lower() == r.label_class
    print("Class-label agreement by group:\n", r.groupby("truth").class_ok.mean().round(3))
    print("Single reviewer: descriptive evidence only.")

if RUN_FORTIS: build_fortis_set()
else: print("Fortis review is OFF (RUN_FORTIS = False). Nothing was built.")